# Advanced XGBoost pipeline — Electric Vehicle Purchases

This is a new, self-contained pipeline; it never reads from or writes to the original `XGBoost/` folder. It is designed for the competition's ROC AUC metric and the supplied XGBoost 3.4 / Optuna 5 environment.

Key upgrades over the baseline:

- **Leak-free 10-fold stratified OOF validation.** Every supervised encoding is learned inside each fold; validation labels never enter validation features.
- **Evidence-based feature engineering.** The EDA shows that subsidy, environmental concern, range anxiety, income, and commute dominate. The notebook adds their known synthetic-data score, charger/access interactions, ratios, exact-boundary flags, frequency features, and digit decompositions.
- **No destructive outlier removal.** The income floor (30,000), commute floor (5 km), and high-income/long-commute edges are predictive generation artifacts. Raw values are retained and boundary flags plus winsorized companion features are added.
- **Correct categorical treatment.** Nominal values are one-hot encoded, true ordinals are mapped explicitly, and repeated exact numeric values/interactions receive smoothed cross-fitted target and residual encodings. Arbitrary `LabelEncoder` ordering is removed.
- **Metric-aligned tuning.** Seeded Optuna TPE tunes mean fold ROC AUC with early stopping, fold-level pruning, a resumable SQLite study, and `scale_pos_weight` itself tuned rather than fixed to the class ratio.
- **XGBoost-only diversity.** Tuned, conservative, and recipe-residual XGBoost variants are blended using honest OOF predictions. The final choice between probability and rank blending is made by OOF ROC AUC.


## 1. Imports and reproducible configuration

The defaults favor score over speed. For a smoke test, reduce `N_TRIALS`, `TUNING_FOLDS`, and `FINAL_FOLDS`; restore the defaults for the real submission. Set `DEVICE = 'cuda'` only when a CUDA-capable XGBoost runtime is available. The default deletes only this notebook's prior study for a deterministic rerun; set `RESUME_STUDY = True` to continue an interrupted search (which adds `N_TRIALS` new trials).


In [1]:
from __future__ import annotations

import gc
import json
import os
import random
import re
import warnings
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import sklearn
import xgboost as xgb
from IPython.display import display
from scipy.special import ndtr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)

# Reproducibility
GLOBAL_SEED = 20260915
os.environ['PYTHONHASHSEED'] = str(GLOBAL_SEED)
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# Score-first defaults. The baseline used 50 x 5 fits; this uses 60 x 3 for
# tuning, then a stronger 10-fold, three-member XGBoost-only ensemble.
RUN_OPTUNA = True
RESUME_STUDY = False
N_TRIALS = 60
TUNING_FOLDS = 3
FINAL_FOLDS = 10
INNER_TE_FOLDS = 5
MAX_BOOST_ROUNDS = 6_000
EARLY_STOPPING_ROUNDS = 200
MAX_BIN = 512
DEVICE = 'cpu'  # Change to 'cuda' only if your local XGBoost has CUDA support.
NTHREAD = -1
BLEND_GRID_STEP = 0.025

print({
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'sklearn': sklearn.__version__,
    'xgboost': xgb.__version__,
    'optuna': optuna.__version__,
    'device': DEVICE,
})


{'numpy': '2.5.2', 'pandas': '3.0.5', 'sklearn': '1.9.0', 'xgboost': '3.4.1', 'optuna': '5.0.0', 'device': 'cpu'}


## 2. Robust paths, loading, and data-contract checks

Paths work whether Jupyter starts in the repository root or in this notebook's folder. Assertions fail early if the competition schema changes, IDs do not align, or target labels are unexpected. All generated artifacts stay under `XGBoost_Advanced/`.


In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'playground-series-s6e9' / 'train.csv').is_file():
            return candidate
    raise FileNotFoundError('Could not find playground-series-s6e9/train.csv from the current directory.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / 'playground-series-s6e9'
OUTPUT_DIR = PROJECT_ROOT / 'XGBoost_Advanced'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SAMPLE_PATH = DATA_DIR / 'sample_submission.csv'
TARGET = 'Will_Buy_EV'
ID_COL = 'id'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

assert TARGET in train.columns and TARGET not in test.columns
assert ID_COL in train.columns and ID_COL in test.columns
assert train[ID_COL].is_unique and test[ID_COL].is_unique
assert test[ID_COL].equals(sample_submission[ID_COL]), 'Sample-submission IDs are not aligned with test IDs.'
assert set(train[TARGET].dropna().unique()) == {'No', 'Yes'}

y = train[TARGET].map({'No': 0, 'Yes': 1}).astype(np.uint8).to_numpy()
RAW_FEATURES = [c for c in test.columns if c != ID_COL]
assert RAW_FEATURES == [c for c in train.columns if c not in (ID_COL, TARGET)]

print(f'Project root: {PROJECT_ROOT}')
print(f'Train: {train.shape} | Test: {test.shape} | Positive rate: {y.mean():.6f}')


Project root: /home/mounir/Desktop/Electric-Vehicle-Purchases
Train: (668665, 15) | Test: (286571, 14) | Positive rate: 0.174645


## 3. Compact EDA audit that drives the pipeline

This reproduces the useful parts of the supplied EDA in tables: missingness, duplicate feature rows, train/test category drift, class rates, and boundary masses. It deliberately avoids fitting a model.


In [3]:
categorical_cols = train[RAW_FEATURES].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
numeric_cols = [c for c in RAW_FEATURES if c not in categorical_cols]

audit = pd.DataFrame({
    'train_dtype': train[RAW_FEATURES].dtypes.astype(str),
    'train_missing': train[RAW_FEATURES].isna().sum(),
    'test_missing': test[RAW_FEATURES].isna().sum(),
    'train_unique': train[RAW_FEATURES].nunique(dropna=False),
    'test_unique': test[RAW_FEATURES].nunique(dropna=False),
})
display(audit)
print('Duplicate feature rows (excluding id/target):', train[RAW_FEATURES].duplicated().sum())

for col in categorical_cols:
    rates = pd.crosstab(train[col], train[TARGET], normalize='index').rename_axis(index=col)
    print(f'\nTarget rates for {col}:')
    display(rates.round(6))

boundary_audit = pd.Series({
    'income_at_30000': int((train['Annual_Income_USD'] == 30_000).sum()),
    'commute_at_5': int((train['Daily_Commute_km'] == 5).sum()),
    'income_at_least_170000': int((train['Annual_Income_USD'] >= 170_000).sum()),
    'long_commute_at_least_80': int((train['Daily_Commute_km'] >= 80).sum()),
})
display(boundary_audit.to_frame('train_count'))


,train_dtype,train_missing,test_missing,train_unique,test_unique
Age,int64,0,0,45,45
Annual_Income_USD,float64,0,0,13214,10526
Daily_Commute_km,float64,0,0,805,781
Number_of_Cars_Owned,int64,0,0,4,4
Charging_Stations_Near_Home,int64,0,0,15,15
Charging_Stations_Near_Work,int64,0,0,20,20
Environmental_Concern_Level,float64,0,0,5,5
Gender,str,0,0,3,3
City_Type,str,0,0,3,3
Current_Car_Type,str,0,0,4,4


Duplicate feature rows (excluding id/target): 0

Target rates for Gender:


Will_Buy_EV,No,Yes
Gender,,
Female,0.822359,0.177641
Male,0.827747,0.172253
Other,0.826268,0.173732



Target rates for City_Type:


Will_Buy_EV,No,Yes
City_Type,,
Rural,0.806611,0.193389
Suburban,0.819064,0.180936
Urban,0.838942,0.161058


...


Target rates for Subsidy_Available:


Will_Buy_EV,No,Yes
Subsidy_Available,,
No,0.994243,0.005757
Yes,0.725305,0.274695



Target rates for Range_Anxiety_Level:


Will_Buy_EV,No,Yes
Range_Anxiety_Level,,
High,0.998633,0.001367
Low,0.810973,0.189027
Medium,0.958255,0.041745


,train_count
income_at_30000,61605
commute_at_5,144280
income_at_least_170000,393
long_commute_at_least_80,219


## 4. Unsupervised and domain/synthetic feature engineering

The raw columns remain present. Winsorized columns are only companion views, not replacements. Combined train+test frequency maps and category vocabularies use no target labels, so this transductive step cannot leak validation outcomes. The recipe feature reflects the documented structure of the original synthetic source: income, environmental concern, subsidy, and range-anxiety penalties.


In [4]:
NUMERIC_COLS = [
    'Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
    'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
    'Environmental_Concern_Level',
]
CATEGORICAL_COLS = [
    'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
    'Subsidy_Available', 'Range_Anxiety_Level',
]
assert set(NUMERIC_COLS + CATEGORICAL_COLS) == set(RAW_FEATURES)

train_medians = train[NUMERIC_COLS].median()
clip_bounds = {
    col: tuple(train[col].quantile([0.005, 0.995]).astype(float))
    for col in ('Annual_Income_USD', 'Daily_Commute_km')
}


def clean_raw(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame[RAW_FEATURES].copy()
    for col in NUMERIC_COLS:
        out[col] = pd.to_numeric(out[col], errors='coerce').fillna(train_medians[col])
    for col in CATEGORICAL_COLS:
        out[col] = out[col].astype('string').fillna('__MISSING__').astype(str)
    return out


train_raw = clean_raw(train)
test_raw = clean_raw(test)
all_raw = pd.concat([train_raw, test_raw], axis=0, ignore_index=True)

frequency_maps = {
    col: all_raw[col].value_counts(dropna=False, normalize=True)
    for col in RAW_FEATURES
}


def ev_recipe_score(frame: pd.DataFrame) -> np.ndarray:
    subsidy = frame['Subsidy_Available'].eq('Yes').to_numpy(dtype=np.float64)
    anxiety_penalty = frame['Range_Anxiety_Level'].map(
        {'Low': 0.0, 'Medium': -1.0, 'High': -3.0}
    ).fillna(-1.0).to_numpy(dtype=np.float64)
    return (
        1.2 * frame['Annual_Income_USD'].to_numpy(dtype=np.float64) / 100_000.0
        + 0.6 * frame['Environmental_Concern_Level'].to_numpy(dtype=np.float64)
        + 2.0 * subsidy
        + anxiety_penalty
    )


def ev_recipe_probability(frame: pd.DataFrame) -> np.ndarray:
    # The source rule is a probit-style threshold with unit Gaussian noise.
    return np.clip(ndtr(ev_recipe_score(frame) - 5.5), 1e-6, 1.0 - 1e-6)


def ev_recipe_logit_margin(frame: pd.DataFrame) -> np.ndarray:
    probability = ev_recipe_probability(frame)
    return np.log(probability / (1.0 - probability)).astype(np.float32)


def safe_divide(numerator: pd.Series, denominator: pd.Series, offset: float = 1.0) -> np.ndarray:
    return numerator.to_numpy(dtype=np.float64) / (denominator.to_numpy(dtype=np.float64) + offset)


def sanitize_feature_names(columns: list[str]) -> list[str]:
    counts: dict[str, int] = {}
    cleaned: list[str] = []
    for column in columns:
        base = re.sub(r'[^0-9A-Za-z_]+', '_', str(column)).strip('_') or 'feature'
        occurrence = counts.get(base, 0)
        counts[base] = occurrence + 1
        cleaned.append(base if occurrence == 0 else f'{base}_{occurrence}')
    return cleaned


def build_base_features(frame: pd.DataFrame) -> pd.DataFrame:
    x = frame[NUMERIC_COLS].astype(np.float32).copy()
    income = frame['Annual_Income_USD']
    commute = frame['Daily_Commute_km']
    cars = frame['Number_of_Cars_Owned']
    home_stations = frame['Charging_Stations_Near_Home']
    work_stations = frame['Charging_Stations_Near_Work']
    concern = frame['Environmental_Concern_Level']
    subsidy = frame['Subsidy_Available'].eq('Yes').astype(np.float32)
    home_charge = frame['Home_Charging_Possible'].eq('Yes').astype(np.float32)
    anxiety = frame['Range_Anxiety_Level'].map({'Low': 0.0, 'Medium': 1.0, 'High': 2.0}).fillna(1.0)

    # Known ordinal/binary meanings; nominal categories are one-hot encoded below.
    x['Subsidy_flag'] = subsidy
    x['Home_charging_flag'] = home_charge
    x['Range_anxiety_ordinal'] = anxiety.astype(np.float32)

    # Charging convenience and capacity interactions.
    total_stations = home_stations + work_stations
    x['Total_charging_stations'] = total_stations.astype(np.float32)
    x['Station_difference'] = (home_stations - work_stations).astype(np.float32)
    x['Station_minimum'] = np.minimum(home_stations, work_stations).astype(np.float32)
    x['Station_maximum'] = np.maximum(home_stations, work_stations).astype(np.float32)
    x['Station_product'] = (home_stations * work_stations).astype(np.float32)
    x['No_nearby_station'] = total_stations.eq(0).astype(np.float32)
    x['Either_station_zero'] = (home_stations.eq(0) | work_stations.eq(0)).astype(np.float32)
    x['Home_access_score'] = (home_stations + 3.0 * home_charge).astype(np.float32)
    x['Commute_per_station'] = safe_divide(commute, total_stations, offset=1.0).astype(np.float32)
    x['Stations_per_commute'] = safe_divide(total_stations, commute, offset=1.0).astype(np.float32)

    # Affordability / usage ratios and monotone companion transforms.
    x['Log_income'] = np.log1p(income).astype(np.float32)
    x['Log_commute'] = np.log1p(commute).astype(np.float32)
    x['Income_per_commute'] = safe_divide(income, commute, offset=1.0).astype(np.float32)
    x['Income_per_car'] = safe_divide(income, cars, offset=0.0).astype(np.float32)
    x['Income_x_subsidy'] = (income * subsidy).astype(np.float32)
    x['Concern_x_subsidy'] = (concern * subsidy).astype(np.float32)
    x['Concern_minus_anxiety'] = (concern - anxiety).astype(np.float32)
    x['Commute_x_anxiety'] = (commute * (anxiety + 1.0)).astype(np.float32)

    # The raw source-data formula gives trees a one-split summary and a strong prior.
    recipe_score = ev_recipe_score(frame)
    x['EV_recipe_score'] = recipe_score.astype(np.float32)
    x['EV_recipe_distance'] = (recipe_score - 5.5).astype(np.float32)
    x['EV_recipe_probability'] = ev_recipe_probability(frame).astype(np.float32)

    # Preserve raw tails. These companion views/flags expose clipping and hard edges.
    for col, (lower, upper) in clip_bounds.items():
        x[f'{col}_winsorized'] = frame[col].clip(lower, upper).astype(np.float32)
    x['Income_at_floor'] = income.eq(30_000).astype(np.float32)
    x['Commute_at_floor'] = commute.eq(5.0).astype(np.float32)
    x['Income_at_least_170k'] = income.ge(170_000).astype(np.float32)
    x['Commute_at_least_80'] = commute.ge(80.0).astype(np.float32)

    # Exact repeated values are a feature of this synthetic dataset, not measurement noise.
    for col in RAW_FEATURES:
        x[f'{col}_frequency'] = frame[col].map(frequency_maps[col]).fillna(0.0).astype(np.float32)

    # Digit decompositions expose modulo patterns that ordinary threshold splits cannot learn cheaply.
    categorical = frame[CATEGORICAL_COLS].copy()
    categorical['Subsidy_x_anxiety'] = frame['Subsidy_Available'] + '__' + frame['Range_Anxiety_Level']
    categorical['Subsidy_x_concern'] = frame['Subsidy_Available'] + '__' + concern.astype(int).astype(str)
    categorical['City_x_home_charge'] = frame['City_Type'] + '__' + frame['Home_Charging_Possible']
    categorical['City_x_car_type'] = frame['City_Type'] + '__' + frame['Current_Car_Type']

    income_integer = np.rint(income.to_numpy()).astype(np.int64)
    for place, label in ((1, 'ones'), (10, 'tens'), (100, 'hundreds'), (1_000, 'thousands'), (10_000, 'ten_thousands'), (100_000, 'hundred_thousands')):
        categorical[f'Income_digit_{label}'] = ((income_integer // place) % 10).astype(str)
    commute_tenths = np.rint(commute.to_numpy(dtype=np.float64) * 10.0).astype(np.int64)
    categorical['Commute_tenths_digit'] = (commute_tenths % 10).astype(str)
    categorical['Commute_ones_digit'] = ((commute_tenths // 10) % 10).astype(str)
    categorical['Commute_tens_digit'] = ((commute_tenths // 100) % 10).astype(str)
    age_integer = frame['Age'].to_numpy(dtype=np.int64)
    categorical['Age_ones_digit'] = (age_integer % 10).astype(str)

    one_hot = pd.get_dummies(categorical, prefix=categorical.columns, dtype=np.int8)
    x = pd.concat([x, one_hot], axis=1)
    x.columns = sanitize_feature_names(x.columns.astype(str).tolist())
    x = x.astype(np.float32)
    assert np.isfinite(x.to_numpy()).all(), 'Feature engineering produced NaN or infinity.'
    return x


all_base_features = build_base_features(all_raw)
BASE_FEATURE_NAMES = all_base_features.columns.tolist()
X_TRAIN_BASE = np.ascontiguousarray(all_base_features.iloc[: len(train)].to_numpy(dtype=np.float32))
X_TEST_BASE = np.ascontiguousarray(all_base_features.iloc[len(train):].to_numpy(dtype=np.float32))
del all_base_features, all_raw
gc.collect()

print(f'Base engineered matrix: {X_TRAIN_BASE.shape[1]} features')
print(f'Train matrix memory: {X_TRAIN_BASE.nbytes / 2**20:.1f} MiB')


Base engineered matrix: 193 features
Train matrix memory: 492.3 MiB


## 5. Fold-safe smoothed target and residual encoding

Income has only about 13k exact values and commute about 800 across 669k rows, so exact values repeat heavily. A tree sees them as ordered numbers and cannot efficiently learn reproducible, non-monotonic generator artifacts. These encoders add that view. Training rows are encoded by an inner stratified split; outer validation/test rows are mapped using outer-training labels only. Residual encodings estimate corrections to the synthetic recipe rather than relearning its dominant effects.


In [5]:
# name, grouping columns, smoothing strength, add recipe-residual encoding
TE_SPECS = [
    ('income_exact', ['Annual_Income_USD'], 80.0, True),
    ('commute_exact', ['Daily_Commute_km'], 150.0, True),
    ('income_subsidy_concern', ['Annual_Income_USD', 'Subsidy_Available', 'Environmental_Concern_Level'], 100.0, True),
    ('commute_anxiety', ['Daily_Commute_km', 'Range_Anxiety_Level'], 150.0, True),
    ('subsidy_concern_anxiety', ['Subsidy_Available', 'Environmental_Concern_Level', 'Range_Anxiety_Level'], 500.0, False),
    ('city_home', ['City_Type', 'Home_Charging_Possible'], 500.0, False),
    ('charger_pair', ['Charging_Stations_Near_Home', 'Charging_Stations_Near_Work'], 250.0, False),
    ('age_gender', ['Age', 'Gender'], 250.0, False),
]

TE_FEATURE_NAMES = []
for name, _, _, add_residual in TE_SPECS:
    TE_FEATURE_NAMES.append(f'TE_{name}')
    if add_residual:
        TE_FEATURE_NAMES.append(f'TE_recipe_residual_{name}')
ALL_FEATURE_NAMES = BASE_FEATURE_NAMES + TE_FEATURE_NAMES


def smoothed_group_mean(
    fit_frame: pd.DataFrame,
    fit_values: np.ndarray,
    apply_frame: pd.DataFrame,
    columns: list[str],
    smoothing: float,
) -> np.ndarray:
    values = np.asarray(fit_values, dtype=np.float64)
    prior = float(values.mean())
    stats_frame = fit_frame[columns].copy()
    stats_frame['_encoding_value'] = values
    stats = stats_frame.groupby(columns, sort=False, dropna=False)['_encoding_value'].agg(['sum', 'count'])
    encoded_map = (stats['sum'] + smoothing * prior) / (stats['count'] + smoothing)

    if len(columns) == 1:
        encoded = apply_frame[columns[0]].map(encoded_map).to_numpy(dtype=np.float64)
    else:
        apply_index = pd.MultiIndex.from_frame(apply_frame[columns], names=columns)
        encoded = encoded_map.reindex(apply_index).to_numpy(dtype=np.float64)
    return np.nan_to_num(encoded, nan=prior).astype(np.float32)


def build_fold_target_encodings(
    fit_frame: pd.DataFrame,
    fit_y: np.ndarray,
    apply_frames: list[pd.DataFrame],
    seed: int,
) -> tuple[np.ndarray, list[np.ndarray]]:
    fit_frame = fit_frame.reset_index(drop=True)
    apply_frames = [frame.reset_index(drop=True) for frame in apply_frames]
    fit_y = np.asarray(fit_y, dtype=np.float64)
    residual_y = fit_y - ev_recipe_probability(fit_frame)

    inner_cv = StratifiedKFold(n_splits=INNER_TE_FOLDS, shuffle=True, random_state=seed)
    inner_splits = list(inner_cv.split(np.zeros(len(fit_y)), fit_y.astype(np.uint8)))
    fit_columns: list[np.ndarray] = []
    apply_columns: list[list[np.ndarray]] = [[] for _ in apply_frames]

    for _, columns, smoothing, add_residual in TE_SPECS:
        value_sets = [fit_y]
        if add_residual:
            value_sets.append(residual_y)

        for values in value_sets:
            cross_fitted = np.empty(len(fit_frame), dtype=np.float32)
            for inner_train_idx, inner_valid_idx in inner_splits:
                cross_fitted[inner_valid_idx] = smoothed_group_mean(
                    fit_frame.iloc[inner_train_idx],
                    values[inner_train_idx],
                    fit_frame.iloc[inner_valid_idx],
                    columns,
                    smoothing,
                )
            fit_columns.append(cross_fitted)

            for apply_number, apply_frame in enumerate(apply_frames):
                apply_columns[apply_number].append(
                    smoothed_group_mean(fit_frame, values, apply_frame, columns, smoothing)
                )

    fit_encoded = np.column_stack(fit_columns).astype(np.float32)
    apply_encoded = [np.column_stack(columns).astype(np.float32) for columns in apply_columns]
    assert fit_encoded.shape[1] == len(TE_FEATURE_NAMES)
    return fit_encoded, apply_encoded


print(f'Target-encoded features per fold: {len(TE_FEATURE_NAMES)}')
print(f'Total model features: {len(ALL_FEATURE_NAMES)}')


Target-encoded features per fold: 12
Total model features: 205


## 6. Optuna preparation and metric-aligned search

The tuning fold matrices are constructed once so trials compare identical data and encodings. `QuantileDMatrix` saves memory for histogram training. TPE explores capacity, regularization, sampling, and a modest class-weight range; the original fixed 4.73 class ratio is not assumed optimal for a ranking metric. Early stopping keeps only the best iteration, and fold-level median pruning terminates weak trials.


In [6]:
def join_features(base_features: np.ndarray, encoded_features: np.ndarray) -> np.ndarray:
    return np.ascontiguousarray(np.column_stack([base_features, encoded_features]), dtype=np.float32)


def make_quantile_dmatrix(
    features: np.ndarray,
    labels: np.ndarray | None = None,
    reference: xgb.QuantileDMatrix | None = None,
    base_margin: np.ndarray | None = None,
) -> xgb.QuantileDMatrix:
    return xgb.QuantileDMatrix(
        features,
        label=labels,
        base_margin=base_margin,
        feature_names=ALL_FEATURE_NAMES,
        max_bin=MAX_BIN,
        ref=reference,
        nthread=NTHREAD,
    )


def materialize_xgb_params(search_params: dict, seed: int) -> dict:
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'booster': 'gbtree',
        'tree_method': 'hist',
        'device': DEVICE,
        'max_bin': MAX_BIN,
        'nthread': NTHREAD,
        'seed': seed,
        'verbosity': 0,
        'validate_parameters': True,
        **search_params,
    }
    if params['grow_policy'] == 'depthwise':
        params['max_leaves'] = 0
    else:
        params['max_depth'] = 0
    return params


def fit_one_booster(params: dict, dtrain: xgb.QuantileDMatrix, dvalid: xgb.QuantileDMatrix) -> xgb.Booster:
    early_stopping = xgb.callback.EarlyStopping(
        rounds=EARLY_STOPPING_ROUNDS,
        metric_name='auc',
        data_name='valid',
        maximize=True,
        save_best=True,
    )
    return xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=MAX_BOOST_ROUNDS,
        evals=[(dvalid, 'valid')],
        callbacks=[early_stopping],
        verbose_eval=False,
    )


def predict_best(model: xgb.Booster, data: xgb.QuantileDMatrix) -> np.ndarray:
    # Explicitly guard against the native xgb.train last-iteration pitfall.
    end_iteration = int(model.best_iteration) + 1
    return model.predict(data, iteration_range=(0, end_iteration)).astype(np.float64)


FALLBACK_SEARCH_PARAMS = {
    'learning_rate': 0.035,
    'grow_policy': 'depthwise',
    'max_depth': 5,
    'min_child_weight': 18.0,
    'subsample': 0.90,
    'colsample_bytree': 0.82,
    'colsample_bynode': 0.90,
    'reg_alpha': 0.08,
    'reg_lambda': 5.0,
    'gamma': 0.02,
    'max_delta_step': 0,
    'scale_pos_weight': 1.0,
}


def suggest_search_params(trial: optuna.Trial) -> dict:
    grow_policy = trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide'])
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.012, 0.080, log=True),
        'grow_policy': grow_policy,
        'min_child_weight': trial.suggest_float('min_child_weight', 2.0, 100.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.65, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.55, 1.0),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.55, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 30.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-4, 2.0, log=True),
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 4),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.70, 2.50, log=True),
    }
    if grow_policy == 'depthwise':
        params['max_depth'] = trial.suggest_int('max_depth', 3, 8)
    else:
        params['max_leaves'] = trial.suggest_int('max_leaves', 16, 160, log=True)
    return params


In [7]:
tuning_cv = StratifiedKFold(n_splits=TUNING_FOLDS, shuffle=True, random_state=GLOBAL_SEED)
TUNING_MATRICES: list[tuple[xgb.QuantileDMatrix, xgb.QuantileDMatrix, np.ndarray]] = []

for fold, (train_idx, valid_idx) in enumerate(tuning_cv.split(X_TRAIN_BASE, y), start=1):
    encoded_train, (encoded_valid,) = build_fold_target_encodings(
        train_raw.iloc[train_idx],
        y[train_idx],
        [train_raw.iloc[valid_idx]],
        seed=GLOBAL_SEED + fold,
    )
    fold_train = join_features(X_TRAIN_BASE[train_idx], encoded_train)
    fold_valid = join_features(X_TRAIN_BASE[valid_idx], encoded_valid)
    dtrain = make_quantile_dmatrix(fold_train, labels=y[train_idx])
    dvalid = make_quantile_dmatrix(fold_valid, labels=y[valid_idx], reference=dtrain)
    TUNING_MATRICES.append((dtrain, dvalid, y[valid_idx]))
    del encoded_train, encoded_valid, fold_train, fold_valid
    gc.collect()
    print(f'Cached tuning fold {fold}/{TUNING_FOLDS}')


Cached tuning fold 1/3
Cached tuning fold 2/3
Cached tuning fold 3/3


In [8]:
def objective(trial: optuna.Trial) -> float:
    search_params = suggest_search_params(trial)
    fold_scores: list[float] = []
    fold_iterations: list[int] = []

    for fold, (dtrain, dvalid, valid_y) in enumerate(TUNING_MATRICES):
        params = materialize_xgb_params(search_params, seed=GLOBAL_SEED + 1_000 * trial.number + fold)
        model = fit_one_booster(params, dtrain, dvalid)
        predictions = predict_best(model, dvalid)
        fold_scores.append(float(roc_auc_score(valid_y, predictions)))
        fold_iterations.append(int(model.best_iteration) + 1)
        trial.report(float(np.mean(fold_scores)), step=fold)
        del model, predictions
        gc.collect()
        if trial.should_prune():
            raise optuna.TrialPruned()

    trial.set_user_attr('fold_auc', fold_scores)
    trial.set_user_attr('fold_auc_std', float(np.std(fold_scores)))
    trial.set_user_attr('best_iterations', fold_iterations)
    return float(np.mean(fold_scores))


STUDY_DB = OUTPUT_DIR / 'xgboost_advanced_optuna.db'
STUDY_NAME = 'ev_xgboost_advanced_v1'
if not RESUME_STUDY and STUDY_DB.exists():
    STUDY_DB.unlink()
storage_url = f'sqlite:///{STUDY_DB.as_posix()}'

if RUN_OPTUNA:
    sampler = optuna.samplers.TPESampler(seed=GLOBAL_SEED, multivariate=True, group=True)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=12, n_warmup_steps=1, n_min_trials=5)
    study = optuna.create_study(
        study_name=STUDY_NAME,
        storage=storage_url,
        direction='maximize',
        sampler=sampler,
        pruner=pruner,
        load_if_exists=RESUME_STUDY,
    )
    if len(study.trials) == 0:
        study.enqueue_trial(FALLBACK_SEARCH_PARAMS)
    study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True, show_progress_bar=True)
    best_search_params = dict(study.best_trial.params)
    study.trials_dataframe().to_csv(OUTPUT_DIR / 'xgboost_advanced_optuna_trials.csv', index=False)
    print(f'Best tuning AUC: {study.best_value:.8f}')
    print('Best search parameters:', json.dumps(best_search_params, indent=2))
else:
    study = None
    best_search_params = dict(FALLBACK_SEARCH_PARAMS)
    print('Optuna skipped; using documented fallback parameters.')

# Release cached tuning matrices before the larger final OOF ensemble.
TUNING_MATRICES.clear()
del dtrain, dvalid
gc.collect()


/tmp/ipykernel_81727/1153054675.py:31: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(seed=GLOBAL_SEED, multivariate=True, group=True)
[I 2026-09-15 17:44:21,481] A new study created in RDB with name: ev_xgboost_advanced_v1


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-09-15 17:47:48,003] Trial 0 finished with value: 0.945174285911578 and parameters: {'grow_policy': 'depthwise', 'learning_rate': 0.035, 'min_child_weight': 18.0, 'subsample': 0.9, 'colsample_bytree': 0.82, 'colsample_bynode': 0.9, 'reg_alpha': 0.08, 'reg_lambda': 5.0, 'gamma': 0.02, 'max_delta_step': 0, 'scale_pos_weight': 1.0, 'max_depth': 5}. Best is trial 0 with value: 0.945174285911578.
[I 2026-09-15 17:50:21,075] Trial 1 finished with value: 0.945086335840925 and parameters: {'grow_policy': 'depthwise', 'learning_rate': 0.037694094682979566, 'min_child_weight': 50.82423800571425, 'subsample': 0.8548195615416869, 'colsample_bytree': 0.8253875352093305, 'colsample_bynode': 0.8193310363234336, 'reg_alpha': 0.036656615564296065, 'reg_lambda': 2.0594334698003713, 'gamma': 0.22131989122907114, 'max_delta_step': 1, 'scale_pos_weight': 1.9150454457965378, 'max_depth': 6}. Best is trial 0 with value: 0.945174285911578.
[I 2026-09-15 17:54:16,175] Trial 2 finished with value: 0.9451

53

## 7. Final 10-fold XGBoost-only ensemble

Three related but non-identical learners create useful ranking diversity: the Optuna winner, a slightly more regularized/lower-capacity version, and a model initialized with the probit recipe converted to XGBoost log-odds margins. Each produces complete OOF and averaged test predictions.


In [9]:
def conservative_variant(search_params: dict) -> dict:
    variant = dict(search_params)
    variant['learning_rate'] = max(0.010, float(variant['learning_rate']) * 0.82)
    variant['min_child_weight'] = min(140.0, float(variant['min_child_weight']) * 1.35)
    variant['reg_lambda'] = min(50.0, float(variant['reg_lambda']) * 1.35)
    variant['subsample'] = min(1.0, max(0.72, float(variant['subsample']) + 0.04))
    if variant['grow_policy'] == 'depthwise':
        variant['max_depth'] = max(3, int(variant['max_depth']) - 1)
    else:
        variant['max_leaves'] = max(12, int(round(int(variant['max_leaves']) * 0.72)))
    return variant


MEMBERS = [
    {'name': 'tuned_direct', 'search_params': dict(best_search_params), 'seed': GLOBAL_SEED + 101, 'base_margin': False},
    {'name': 'conservative_direct', 'search_params': conservative_variant(best_search_params), 'seed': GLOBAL_SEED + 202, 'base_margin': False},
    {'name': 'recipe_residual', 'search_params': dict(best_search_params), 'seed': GLOBAL_SEED + 303, 'base_margin': True},
]
MEMBER_NAMES = [member['name'] for member in MEMBERS]
n_members = len(MEMBERS)
member_oof = np.zeros((n_members, len(train)), dtype=np.float64)
member_test = np.zeros((n_members, len(test)), dtype=np.float64)
fold_records: list[dict] = []
feature_gain_sum: dict[str, float] = {}

final_cv = StratifiedKFold(n_splits=FINAL_FOLDS, shuffle=True, random_state=GLOBAL_SEED + 17)
for fold, (train_idx, valid_idx) in enumerate(final_cv.split(X_TRAIN_BASE, y), start=1):
    print(f'\n===== Final fold {fold}/{FINAL_FOLDS} =====')
    fold_train_raw = train_raw.iloc[train_idx]
    fold_valid_raw = train_raw.iloc[valid_idx]
    encoded_train, (encoded_valid, encoded_test) = build_fold_target_encodings(
        fold_train_raw,
        y[train_idx],
        [fold_valid_raw, test_raw],
        seed=GLOBAL_SEED + 10_000 + fold,
    )
    fold_train = join_features(X_TRAIN_BASE[train_idx], encoded_train)
    fold_valid = join_features(X_TRAIN_BASE[valid_idx], encoded_valid)
    fold_test = join_features(X_TEST_BASE, encoded_test)

    # Build once and reuse for both direct learners.
    dtrain = make_quantile_dmatrix(fold_train, labels=y[train_idx])
    dvalid = make_quantile_dmatrix(fold_valid, labels=y[valid_idx], reference=dtrain)
    dtest = make_quantile_dmatrix(fold_test, reference=dtrain)

    for member_idx, member in enumerate(MEMBERS[:2]):
        params = materialize_xgb_params(member['search_params'], seed=member['seed'] + fold)
        model = fit_one_booster(params, dtrain, dvalid)
        valid_predictions = predict_best(model, dvalid)
        test_predictions = predict_best(model, dtest)
        member_oof[member_idx, valid_idx] = valid_predictions
        member_test[member_idx] += test_predictions / FINAL_FOLDS
        fold_auc = float(roc_auc_score(y[valid_idx], valid_predictions))
        fold_records.append({
            'member': member['name'], 'fold': fold, 'auc': fold_auc,
            'best_iteration': int(model.best_iteration) + 1,
        })
        for feature, gain in model.get_score(importance_type='gain').items():
            feature_gain_sum[feature] = feature_gain_sum.get(feature, 0.0) + float(gain)
        print(f"{member['name']}: AUC={fold_auc:.8f}, trees={int(model.best_iteration) + 1}")
        del model, valid_predictions, test_predictions
        gc.collect()

    # The third learner starts from the source-recipe prior and learns residual corrections.
    recipe_member = MEMBERS[2]
    recipe_params = materialize_xgb_params(recipe_member['search_params'], seed=recipe_member['seed'] + fold)
    recipe_params['scale_pos_weight'] = 1.0
    dtrain.set_base_margin(ev_recipe_logit_margin(fold_train_raw))
    dvalid.set_base_margin(ev_recipe_logit_margin(fold_valid_raw))
    dtest.set_base_margin(ev_recipe_logit_margin(test_raw))
    recipe_model = fit_one_booster(recipe_params, dtrain, dvalid)
    recipe_valid_predictions = predict_best(recipe_model, dvalid)
    recipe_test_predictions = predict_best(recipe_model, dtest)
    member_oof[2, valid_idx] = recipe_valid_predictions
    member_test[2] += recipe_test_predictions / FINAL_FOLDS
    recipe_auc = float(roc_auc_score(y[valid_idx], recipe_valid_predictions))
    fold_records.append({
        'member': recipe_member['name'], 'fold': fold, 'auc': recipe_auc,
        'best_iteration': int(recipe_model.best_iteration) + 1,
    })
    for feature, gain in recipe_model.get_score(importance_type='gain').items():
        feature_gain_sum[feature] = feature_gain_sum.get(feature, 0.0) + float(gain)
    print(f"{recipe_member['name']}: AUC={recipe_auc:.8f}, trees={int(recipe_model.best_iteration) + 1}")

    del (
        encoded_train, encoded_valid, encoded_test, fold_train, fold_valid, fold_test,
        dtrain, dvalid, dtest, recipe_model, recipe_valid_predictions, recipe_test_predictions,
    )
    gc.collect()

print('\nIndividual full-OOF scores:')
for member_idx, member_name in enumerate(MEMBER_NAMES):
    print(f'{member_name}: {roc_auc_score(y, member_oof[member_idx]):.8f}')



===== Final fold 1/10 =====
tuned_direct: AUC=0.94381539, trees=3598
conservative_direct: AUC=0.94378635, trees=5992
recipe_residual: AUC=0.94386049, trees=3401

===== Final fold 2/10 =====
tuned_direct: AUC=0.94678280, trees=4233
conservative_direct: AUC=0.94669306, trees=5763
recipe_residual: AUC=0.94679938, trees=2956

===== Final fold 3/10 =====
tuned_direct: AUC=0.94555241, trees=2731
conservative_direct: AUC=0.94547846, trees=5268
recipe_residual: AUC=0.94554231, trees=2306

===== Final fold 4/10 =====
tuned_direct: AUC=0.94651602, trees=3602
conservative_direct: AUC=0.94650771, trees=5998
recipe_residual: AUC=0.94657795, trees=3196

===== Final fold 5/10 =====
tuned_direct: AUC=0.94581789, trees=2784
conservative_direct: AUC=0.94577575, trees=4737
recipe_residual: AUC=0.94586922, trees=2995

===== Final fold 6/10 =====
tuned_direct: AUC=0.94514456, trees=3015
conservative_direct: AUC=0.94511669, trees=5910
recipe_residual: AUC=0.94521799, trees=2685

===== Final fold 7/10 =====

## 8. OOF blend selection and competition artifacts

ROC AUC depends on ordering. We therefore compare ordinary probability averaging with percentile-rank averaging on complete OOF predictions. A small non-negative simplex grid chooses weights; with 669k OOF rows and only three members this is much less prone to weight overfitting than a large unconstrained stack. The higher-OOF method is applied identically to test predictions.


In [10]:
def percentile_ranks(predictions: np.ndarray) -> np.ndarray:
    ranked = np.empty_like(predictions, dtype=np.float64)
    for row in range(predictions.shape[0]):
        ranked[row] = pd.Series(predictions[row]).rank(method='average', pct=True).to_numpy()
    return ranked


def optimize_three_way_blend(predictions: np.ndarray, labels: np.ndarray, step: float) -> tuple[np.ndarray, float]:
    units = int(round(1.0 / step))
    best_weights = np.array([1.0, 0.0, 0.0], dtype=np.float64)
    best_auc = -np.inf
    for first in range(units + 1):
        for second in range(units - first + 1):
            weights = np.array([first, second, units - first - second], dtype=np.float64) / units
            auc = float(roc_auc_score(labels, weights @ predictions))
            if auc > best_auc:
                best_auc = auc
                best_weights = weights
    return best_weights, best_auc


probability_weights, probability_auc = optimize_three_way_blend(member_oof, y, BLEND_GRID_STEP)
rank_oof = percentile_ranks(member_oof)
rank_test = percentile_ranks(member_test)
rank_weights, rank_auc = optimize_three_way_blend(rank_oof, y, BLEND_GRID_STEP)

if rank_auc > probability_auc:
    chosen_method = 'rank'
    chosen_weights = rank_weights
    final_oof = rank_weights @ rank_oof
    final_test = rank_weights @ rank_test
    final_auc = rank_auc
else:
    chosen_method = 'probability'
    chosen_weights = probability_weights
    final_oof = probability_weights @ member_oof
    final_test = probability_weights @ member_test
    final_auc = probability_auc

final_test = np.clip(final_test, 0.0, 1.0)
assert np.isfinite(final_oof).all() and np.isfinite(final_test).all()
assert len(final_test) == len(test)

print(f'Probability blend: AUC={probability_auc:.8f}, weights={dict(zip(MEMBER_NAMES, probability_weights.round(3)))}')
print(f'Rank blend:        AUC={rank_auc:.8f}, weights={dict(zip(MEMBER_NAMES, rank_weights.round(3)))}')
print(f'Chosen submission blend: {chosen_method}, OOF AUC={final_auc:.8f}')

# Submission
submission = sample_submission.copy()
submission[TARGET] = final_test
assert submission[ID_COL].equals(test[ID_COL])
assert submission[TARGET].between(0.0, 1.0).all()
submission_path = OUTPUT_DIR / 'submission_xgboost_advanced.csv'
submission.to_csv(submission_path, index=False)

# OOF and per-member test predictions enable later diagnostics or safe ensembling.
oof_output = pd.DataFrame({ID_COL: train[ID_COL], TARGET: y, 'oof_blend': final_oof})
for member_idx, member_name in enumerate(MEMBER_NAMES):
    oof_output[f'oof_{member_name}'] = member_oof[member_idx]
oof_output.to_csv(OUTPUT_DIR / 'xgboost_advanced_oof_predictions.csv', index=False)

test_member_output = pd.DataFrame({ID_COL: test[ID_COL]})
for member_idx, member_name in enumerate(MEMBER_NAMES):
    test_member_output[f'prediction_{member_name}'] = member_test[member_idx]
test_member_output['prediction_blend'] = final_test
test_member_output.to_csv(OUTPUT_DIR / 'xgboost_advanced_test_member_predictions.csv', index=False)

fold_metrics = pd.DataFrame(fold_records)
fold_metrics.to_csv(OUTPUT_DIR / 'xgboost_advanced_fold_metrics.csv', index=False)
display(fold_metrics.groupby('member')['auc'].agg(['mean', 'std', 'min', 'max']).round(8))

trained_model_count = FINAL_FOLDS * len(MEMBERS)
importance = pd.DataFrame({
    'feature': list(feature_gain_sum),
    'mean_gain_across_models': [feature_gain_sum[name] / trained_model_count for name in feature_gain_sum],
}).sort_values('mean_gain_across_models', ascending=False)
importance.to_csv(OUTPUT_DIR / 'xgboost_advanced_feature_importance.csv', index=False)
display(importance.head(40))

metadata = {
    'global_seed': GLOBAL_SEED,
    'tuning_folds': TUNING_FOLDS,
    'final_folds': FINAL_FOLDS,
    'inner_target_encoding_folds': INNER_TE_FOLDS,
    'n_trials_requested_this_run': N_TRIALS if RUN_OPTUNA else 0,
    'best_tuning_auc': float(study.best_value) if study is not None else None,
    'best_search_params': best_search_params,
    'member_oof_auc': {name: float(roc_auc_score(y, member_oof[i])) for i, name in enumerate(MEMBER_NAMES)},
    'probability_blend_auc': probability_auc,
    'probability_weights': dict(zip(MEMBER_NAMES, probability_weights.tolist())),
    'rank_blend_auc': rank_auc,
    'rank_weights': dict(zip(MEMBER_NAMES, rank_weights.tolist())),
    'chosen_method': chosen_method,
    'chosen_weights': dict(zip(MEMBER_NAMES, chosen_weights.tolist())),
    'chosen_oof_auc': final_auc,
    'versions': {
        'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__,
        'xgboost': xgb.__version__, 'optuna': optuna.__version__,
    },
}
with open(OUTPUT_DIR / 'xgboost_advanced_run_metadata.json', 'w', encoding='utf-8') as file:
    json.dump(metadata, file, indent=2)

print(f'\nSaved ready-to-submit file: {submission_path}')
print(f'Rows: {len(submission):,} | OOF AUC: {final_auc:.8f}')
display(submission.head())


Probability blend: AUC=0.94573347, weights={'tuned_direct': np.float64(0.275), 'conservative_direct': np.float64(0.0), 'recipe_residual': np.float64(0.725)}
Rank blend:        AUC=0.94573396, weights={'tuned_direct': np.float64(0.3), 'conservative_direct': np.float64(0.025), 'recipe_residual': np.float64(0.675)}
Chosen submission blend: rank, OOF AUC=0.94573396


,mean,std,min,max
member,,,,
conservative_direct,0.945653,0.000829,0.943786,0.946693
recipe_residual,0.945725,0.000833,0.943860,0.946799
tuned_direct,0.945682,0.000833,0.943815,0.946783


,feature,mean_gain_across_models
30,EV_recipe_probability,2079.720812
29,EV_recipe_distance,1778.030107
28,EV_recipe_score,1684.543729
76,Subsidy_x_concern_Yes__5,1327.978765
25,Concern_x_subsidy,1322.316077
192,TE_subsidy_concern_anxiety,884.508427
7,Subsidy_flag,464.166930
45,Subsidy_Available_frequency,457.222629
26,Concern_minus_anxiety,421.488325
75,Subsidy_x_concern_Yes__4,310.339427



Saved ready-to-submit file: /home/mounir/Desktop/Electric-Vehicle-Purchases/XGBoost_Advanced/submission_xgboost_advanced.csv
Rows: 286,571 | OOF AUC: 0.94573396


,id,Will_Buy_EV
0,668665,0.614745
1,668666,0.500433
2,668667,0.286705
3,668668,0.268098
4,668669,0.559466


## Reading the result

Use `chosen_oof_auc` in `xgboost_advanced_run_metadata.json` as the local headline score and inspect fold standard deviation before trusting tiny gains. The public leaderboard evaluates only about 20% of the test set, so prefer repeatable OOF improvements over differences of a few ten-thousandths on one submission. The generated submission is `XGBoost_Advanced/submission_xgboost_advanced.csv`.
